# 8. Recommendation

This stage does not hardcode a preferred story. It applies three evidence
gates:

1. collaboration must meaningfully distinguish the commercial-plan proxy;
2. existing priority must not already distinguish collaboration reach;
3. the simulated workload must remain limited.

If these conditions fail after a future data refresh, the recommendation
changes to revise or reject the pilot — the action stays tied to evidence,
not to presentation preference.

In [1]:
from common import ARTIFACTS, TIER_ORDER
import json
import pandas as pd
from IPython.display import Markdown, display

customer_view = pd.read_parquet(ARTIFACTS / "customer_view.parquet")
ticket_view = pd.read_parquet(ARTIFACTS / "ticket_view.parquet")
with open(ARTIFACTS / "model_results.json") as f:
    model_results = json.load(f)
with open(ARTIFACTS / "routing_results.json") as f:
    routing_results = json.load(f)

commercial_auc = model_results["commercial_auc"]
priority_auc = model_results["priority_auc"]
moved_count = routing_results["moved_count"]
moved_share = routing_results["moved_share"]
print(f"commercial_auc={commercial_auc:.3f}  priority_auc={priority_auc:.3f}  moved_share={moved_share:.2%}")

commercial_auc=0.704  priority_auc=0.509  moved_share=1.48%


## 8.1 Recompute Tier Aggregates Needed for the Decision

In [2]:
tier_customer = customer_view.groupby("collaboration_tier", observed=True).agg(
    customers=("customer_id", "size"),
    non_free_rate=("non_free_plan", "mean"),
).reindex(TIER_ORDER)
tier_customer["customer_share"] = tier_customer["customers"] / len(customer_view)

tier_ticket = ticket_view.groupby("collaboration_tier", observed=True).agg(
    tickets=("ticket_id", "size"),
    high_priority_rate=("high_priority", "mean"),
).reindex(TIER_ORDER)

hub_customer_count = int(tier_customer.loc["Hub", "customers"])
hub_customer_share = tier_customer.loc["Hub", "customer_share"]
hub_non_free_rate = tier_customer.loc["Hub", "non_free_rate"]
standard_non_free_rate = tier_customer.loc["Standard", "non_free_rate"]
hub_priority_rate = tier_ticket.loc["Hub", "high_priority_rate"]
standard_priority_rate = tier_ticket.loc["Standard", "high_priority_rate"]
print("Tier aggregates recomputed from saved customer_view / ticket_view.")

Tier aggregates recomputed from saved customer_view / ticket_view.


## 8.2 Apply the Evidence Gates and Decide

In [3]:
if commercial_auc >= .60 and priority_auc <= .55 and moved_share <= .10:
    recommendation = "Pilot the collaboration-impact layer on a limited eligible population."
    evidence_statement = (
        "Collaboration reach is associated with the non-Free plan proxy, "
        "while current technical priority does not clearly encode that reach."
    )
elif priority_auc >= .60:
    recommendation = "Do not pilot this uplift as designed."
    evidence_statement = (
        "Current technical priority already reflects collaboration reach strongly enough "
        "to undermine the blind-spot story."
    )
else:
    recommendation = "Collect better timing evidence or revise the policy before a pilot."
    evidence_statement = "The two required relationships are not jointly clear enough."

print(f"Decision: {recommendation}")

Decision: Pilot the collaboration-impact layer on a limited eligible population.


In [4]:
display(Markdown(f'''### Decision

**{recommendation}**

- Hub contains **{hub_customer_count:,}/{len(customer_view):,} customers ({hub_customer_share:.1%})**.
- Hub is **{hub_non_free_rate:.1%} non-Free**, versus **{standard_non_free_rate:.1%}** for Standard reach.
- High/Critical priority covers **{hub_priority_rate:.1%} of Hub tickets
  (n={int(tier_ticket.loc["Hub", "tickets"]):,})**, versus
  **{standard_priority_rate:.1%} of Standard-reach tickets
  (n={int(tier_ticket.loc["Standard", "tickets"]):,})**.
- Model evidence: **{commercial_auc:.3f} commercial AUC** and
  **{priority_auc:.3f} grouped priority AUC**.
- Historical routing workload: **{moved_count:,}/{len(ticket_view):,} tickets ({moved_share:.2%})**.

**Evidence statement:** {evidence_statement}

The recommendation is to test a small routing addition, not to relabel technical severity.
No realised-revenue, churn, observed-SLA or causal-impact claim is supported by this dataset.'''))

### Decision

**Pilot the collaboration-impact layer on a limited eligible population.**

- Hub contains **310/8,320 customers (3.7%)**.
- Hub is **98.7% non-Free**, versus **79.0%** for Standard reach.
- High/Critical priority covers **47.2% of Hub tickets
  (n=322)**, versus
  **50.0% of Standard-reach tickets
  (n=6,862)**.
- Model evidence: **0.704 commercial AUC** and
  **0.509 grouped priority AUC**.
- Historical routing workload: **125/8,469 tickets (1.48%)**.

**Evidence statement:** Collaboration reach is associated with the non-Free plan proxy, while current technical priority does not clearly encode that reach.

The recommendation is to test a small routing addition, not to relabel technical severity.
No realised-revenue, churn, observed-SLA or causal-impact claim is supported by this dataset.

## 8.3 Future Improvements After the Pilot

1. **Improve instrumentation:** capture ticket creation time, routing
   decision, treatment assignment, and account identifier.
2. **Measure service impact:** compare first-response and resolution times
   between treatment and control.
3. **Measure engagement stability:** compare changes in sessions and
   active days using defined pre-ticket and post-ticket windows.
4. **Recalibrate the thresholds:** review the 18, 19, and 20 collaborator
   cut-offs using observed queue capacity and treatment outcomes.
5. **Protect the existing queue:** stop or narrow the policy if
   Critical-ticket service or other customer groups deteriorate.

Revenue enhancement remains a hypothesis. It should only be quantified
after verified contract values and prospective customer outcomes become
available.